# ODI to Databricks Migration: TRG_EMP_INSERT_ONLY

**Conversion Timestamp:** 2024-07-30T12:00:00Z

This notebook contains the converted Spark SQL for an ODI process that performs a direct insert into the `TRG_EMP` table from `EMPLOYEES`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "", "ETL Job Type")
dbutils.widgets.text("DATASOURCE_NUM_ID", "", "Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "", "ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "", "ODI Session Number")

## ETL Parameters

In [ ]:
%sql
CREATE OR REPLACE TEMPORARY VIEW v_etl_parameters AS
SELECT
  '${ETL_JOB_TYPE}' AS etl_job_type,
  '${DATASOURCE_NUM_ID}' AS datasource_num_id,
  '${ETL_PROC_WID}' AS etl_proc_wid,
  '${ODI_SESS_NO}' AS odi_sess_no;

In [ ]:
display(spark.sql("SELECT * FROM v_etl_parameters"))

## Target Table Setup

In [ ]:
%sql
-- SCEN_TASK_NO in {10}: Placeholder for initial setup or DDL checks.
-- SCEN_TASK_NO in {20}: Placeholder for any pre-processing or temp table creation.

CREATE TABLE IF NOT EXISTS workspace.hr.trg_emp (
    EMPLOYEE_ID    BIGINT,
    FIRST_NAME     STRING,
    LAST_NAME      STRING,
    EMAIL          STRING,
    PHONE_NUMBER   STRING,
    HIRE_DATE      TIMESTAMP,
    JOB_ID         STRING,
    SALARY         DECIMAL(10,2),
    COMMISSION_PCT DECIMAL(3,2),
    MANAGER_ID     BIGINT,
    DEPARTMENT_ID  BIGINT
) USING DELTA;

## Data Loading

In [ ]:
%sql
-- SCEN_TASK_NO in {30}: Insert into HR.TRG_EMP

INSERT INTO workspace.hr.trg_emp
(
    EMPLOYEE_ID,
    FIRST_NAME,
    LAST_NAME,
    EMAIL,
    PHONE_NUMBER,
    HIRE_DATE,
    JOB_ID,
    SALARY,
    COMMISSION_PCT,
    MANAGER_ID,
    DEPARTMENT_ID
)
SELECT
    EMPLOYEES.EMPLOYEE_ID,
    EMPLOYEES.FIRST_NAME,
    EMPLOYEES.LAST_NAME,
    EMPLOYEES.EMAIL,
    EMPLOYEES.PHONE_NUMBER,
    EMPLOYEES.HIRE_DATE,
    EMPLOYEES.JOB_ID,
    EMPLOYEES.SALARY,
    EMPLOYEES.COMMISSION_PCT,
    EMPLOYEES.MANAGER_ID,
    EMPLOYEES.DEPARTMENT_ID
FROM
    workspace.hr.employees AS EMPLOYEES;

## Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records_in_trg_emp
FROM workspace.hr.trg_emp;

## Conversion Notes and Manual Actions Required

1.  **Schema and Table Naming:** All schema and table references have been converted to `workspace.hr.<table_name_lowercase>`.
2.  **Oracle Hints Removed:** The `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable to Delta Lake.
3.  **Table DDL:** A `CREATE TABLE IF NOT EXISTS` statement has been added for `workspace.hr.trg_emp` with inferred Spark SQL data types based on common Oracle mappings. You may need to review and adjust these types if they don't exactly match your target schema requirements.
    *   `NUMBER` (integer-like) -> `BIGINT`
    *   `DATE` -> `TIMESTAMP`
    *   `VARCHAR2` -> `STRING`
    *   `NUMBER` (with precision/scale) -> `DECIMAL(p,s)`
4.  **ETL Parameters:** Although no ODI parameters (`#GLOBAL` or `#SCHEMA`) were found in the provided SQL, placeholder widgets and a temporary view for common ETL parameters have been included for consistency with standard ETL notebooks. These can be removed if truly not needed for this specific job.
5.  **SCEN_TASK_NO:** The ODI `SCEN_TASK_NO` markers are preserved as comments within the relevant cells.